# Install mysql-connector-python 

In [1]:
!pip install mysql-connector-python


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import mysql.connector

try:
    mydb = mysql.connector.connect(
        host = "localhost",
        user = "root",
        password = "Suraj$80",
        database = "streaming_project"
    )
    print("SUCCESS!, Python has connected successfully to MySQL")

except mysql.connector.Error as err:
    print(f"Error: {err}")

SUCCESS!, Python has connected successfully to MySQL


## Get data from API (only 20 per page) once.

In [2]:
import requests

# 1. Ask TMDB for today's trending movies
api_key = "765539212f219da3cebb446557cbe329"
url = f"https://api.themoviedb.org/3/trending/movie/day?api_key={api_key}"

print("Fetching live data from TMDB...")
response = requests.get(url)
data = response.json()

# 2. Prepare MySQL to receive the data
cursor = mydb.cursor()

# We use INSERT IGNORE so if you run this script twice, it won't crash from duplicates
insert_query = """
INSERT IGNORE INTO trending_movies (movie_id, title, release_date, popularity, vote_average, vote_count)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# 3. Loop through the JSON data and push it to MySQL
movies_inserted = 0

for movie in data['results']:
    # Sometimes movies don't have a release date yet, this handles that error
    release_date = movie.get('release_date')
    if not release_date:
        release_date = None

    # Map the JSON data to our SQL columns
    values = (
        movie['id'],
        movie.get('title', 'Unknown'),
        release_date,
        movie.get('popularity', 0.0),
        movie.get('vote_average', 0.0),
        movie.get('vote_count', 0)
    )
    
    cursor.execute(insert_query, values)
    movies_inserted += 1

# 4. CRITICAL: Commit the changes to the database
mydb.commit()

print(f"SUCCESS! {movies_inserted} trending movies have been securely loaded into your MySQL database.")

Fetching live data from TMDB...
SUCCESS! 20 trending movies have been securely loaded into your MySQL database.


## Get data of 1000 movies at once, by waiting 10 seconds after each extraction (20 pages)

In [2]:
import mysql.connector
import requests
import time

# 1. Open the connection
mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="Suraj$80", # Enter your MySQL password
  database="streaming_project"
)
cursor = mydb.cursor()

api_key = "765539212f219da3cebb446557cbe329" # Enter your TMDB API Key

insert_query = """
INSERT IGNORE INTO trending_movies (movie_id, title, release_date, popularity, vote_average, vote_count)
VALUES (%s, %s, %s, %s, %s, %s)
"""

pages_to_fetch = 50
total_inserted = 0

print("Starting Indestructible Data Engine...")

# THE DISGUISE: This tells TMDB we are a normal web browser, not a bot.
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

for page in range(1, pages_to_fetch + 1):
    url = f"https://api.themoviedb.org/3/trending/movie/day?api_key={api_key}&page={page}"
    
    max_retries = 3
    success = False
    
    for attempt in range(max_retries):
        try:
            # We send the request WITH the disguise and a 10-second timeout
            response = requests.get(url, headers=headers, timeout=10) 
            data = response.json()
            success = True
            break # If it works, break out of the retry loop!
            
        # THIS CATCHES ERROR 10054 AND PREVENTS THE CRASH
        except requests.exceptions.RequestException as e:
            print(f"Server hung up (Error 10054) on Page {page}. Retrying in 5 seconds... (Attempt {attempt + 1})")
            time.sleep(5) 
            
    if not success:
        print(f"Skipping Page {page} after 3 failed attempts.")
        continue 
        
    if 'results' in data:
        for movie in data['results']:
            release_date = movie.get('release_date')
            if not release_date:
                release_date = None
                
            values = (
                movie['id'],
                movie.get('title', 'Unknown'),
                release_date,
                movie.get('popularity', 0.0),
                movie.get('vote_average', 0.0),
                movie.get('vote_count', 0)
            )
            
            cursor.execute(insert_query, values)
            total_inserted += 1
            
        mydb.commit()
        print(f"Page {page} safely secured.")
    
    time.sleep(0.5) 

# Safely close the doors
cursor.close()
mydb.close()

print(f"\nPIPELINE COMPLETE: Database connection closed safely.")

Starting Indestructible Data Engine...
Server hung up (Error 10054) on Page 1. Retrying in 5 seconds... (Attempt 1)
Page 1 safely secured.
Page 2 safely secured.
Server hung up (Error 10054) on Page 3. Retrying in 5 seconds... (Attempt 1)
Server hung up (Error 10054) on Page 3. Retrying in 5 seconds... (Attempt 2)
Server hung up (Error 10054) on Page 3. Retrying in 5 seconds... (Attempt 3)
Skipping Page 3 after 3 failed attempts.
Page 4 safely secured.
Page 5 safely secured.
Server hung up (Error 10054) on Page 6. Retrying in 5 seconds... (Attempt 1)
Page 6 safely secured.
Server hung up (Error 10054) on Page 7. Retrying in 5 seconds... (Attempt 1)
Page 7 safely secured.
Server hung up (Error 10054) on Page 8. Retrying in 5 seconds... (Attempt 1)
Page 8 safely secured.
Page 9 safely secured.
Server hung up (Error 10054) on Page 10. Retrying in 5 seconds... (Attempt 1)
Page 10 safely secured.
Server hung up (Error 10054) on Page 11. Retrying in 5 seconds... (Attempt 1)
Page 11 safely se

## Now, write the script for the Skipped pages, only skipped pages should get added to database.

In [4]:
import mysql.connector
import requests
import time

# 1. Open the connection
mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="Suraj$80", # Enter your MySQL password
  database="streaming_project"
)
cursor = mydb.cursor()

api_key = "765539212f219da3cebb446557cbe329" # Enter your TMDB API Key

insert_query = """
INSERT IGNORE INTO trending_movies (movie_id, title, release_date, popularity, vote_average, vote_count)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# THE TARGET LIST: Put every skipped page number inside these brackets
skipped_pages = [3, 12, 18, 26, 27, 35, 45, 48] 
total_inserted = 0

print(f"Deploying Mop-Up Script for pages: {skipped_pages}...")

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Notice we are looping through our specific list now!
for page in skipped_pages:
    url = f"https://api.themoviedb.org/3/trending/movie/day?api_key={api_key}&page={page}"
    
    max_retries = 5 # Increased retries to 5 just to be safe
    success = False
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=10) 
            data = response.json()
            success = True
            break 
            
        except requests.exceptions.RequestException as e:
            print(f"TMDB blocked Page {page}. Retrying in 10 seconds... (Attempt {attempt + 1})")
            time.sleep(10) # Increased the pause to 10 seconds to let the server completely cool down
            
    if not success:
        print(f"Page {page} is completely dead right now. Move on.")
        continue 
        
    if 'results' in data:
        for movie in data['results']:
            release_date = movie.get('release_date')
            if not release_date:
                release_date = None
                
            values = (
                movie['id'],
                movie.get('title', 'Unknown'),
                release_date,
                movie.get('popularity', 0.0),
                movie.get('vote_average', 0.0),
                movie.get('vote_count', 0)
            )
            
            cursor.execute(insert_query, values)
            total_inserted += 1
            
        mydb.commit()
        print(f"Target Page {page} safely secured!")
    
    time.sleep(2) # A longer, polite 2-second pause before the next target

# Safely close the doors
cursor.close()
mydb.close()

print(f"\nMOP-UP COMPLETE: The missing pieces have been added to MySQL.")

Deploying Mop-Up Script for pages: [3, 12, 18, 26, 27, 35, 45, 48]...
Target Page 3 safely secured!
TMDB blocked Page 12. Retrying in 10 seconds... (Attempt 1)
TMDB blocked Page 12. Retrying in 10 seconds... (Attempt 2)
Target Page 12 safely secured!
TMDB blocked Page 18. Retrying in 10 seconds... (Attempt 1)
Target Page 18 safely secured!
TMDB blocked Page 26. Retrying in 10 seconds... (Attempt 1)
Target Page 26 safely secured!
Target Page 27 safely secured!
Target Page 35 safely secured!
Target Page 45 safely secured!
Target Page 48 safely secured!

MOP-UP COMPLETE: The missing pieces have been added to MySQL.


### Successfully extracted data into MySQL Database